# Supervised Learning (Módulo 6) – Classificação de MalwareFamily

Treinam-se vários algoritmos de classificação com o **mesmo pré-processamento** do `data_preparation.ipynb` e comparam-se na **validação**. O conjunto de teste não é usado aqui (fica para a avaliação final, no Módulo 7).

Modelos: Logistic Regression, Decision Tree, Random Forest e XGBoost.
Métrica principal: **F1 macro** (a accuracy engana com classes tão desequilibradas).

## 1. Preparação (igual ao `data_preparation.ipynb`)

In [4]:
%pip install -q kagglehub scikit-learn pandas numpy xgboost

import glob, os, time
import numpy as np
import pandas as pd
import kagglehub

TARGET, GROUP, SEED = "MalwareFamily", "Hash", 42

path = kagglehub.dataset_download(
    "raneemababneh/iot-malware-network-traffic-dataset-cic-ynu-2026"
)
df = pd.read_csv(glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)[0])

# --- Data cleaning ---
df = df.replace([np.inf, -np.inf], np.nan).drop_duplicates().reset_index(drop=True)
constantes = [c for c in df.select_dtypes("number") if df[c].nunique(dropna=True) <= 1]
drop_cols = list(dict.fromkeys([TARGET, GROUP, "Arch", "Tot size", "LLC"] + constantes))

# --- Famílias raras (< MIN_HASHES hashes) juntadas em "Other_rare" ---
MIN_HASHES = 10
h = df.groupby(TARGET)[GROUP].nunique()
df[TARGET] = df[TARGET].replace({r: "Other_rare" for r in h[h < MIN_HASHES].index})

# --- Split 80/10/10, estratificado e por grupos (Hash) ---
from sklearn.model_selection import StratifiedGroupKFold
X, y, groups = df.drop(columns=drop_cols), df[TARGET], df[GROUP]

tr, tmp = next(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(X, y, groups))
X_train, y_train = X.iloc[tr], y.iloc[tr]
X_tmp, y_tmp, g_tmp = X.iloc[tmp], y.iloc[tmp], groups.iloc[tmp]
va, te = next(StratifiedGroupKFold(2, shuffle=True, random_state=SEED).split(X_tmp, y_tmp, g_tmp))
X_val, y_val = X_tmp.iloc[va], y_tmp.iloc[va]
X_test, y_test = X_tmp.iloc[te], y_tmp.iloc[te]   # NÃO usar neste notebook

print("Treino:", len(X_train), "| Validação:", len(X_val), "| Teste:", len(X_test))


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Treino: 589570 | Validação: 72666 | Teste: 72379


## 2. Amostra de treino

Treinar 4 modelos em ~590 mil linhas é lento. Para a **comparação** usa-se uma amostra estratificada do treino (as famílias raras mantêm-se representadas). A validação é sempre completa. Depois de escolhido o melhor modelo, pode treinar-se com o treino todo (`TRAIN_SAMPLE = None`).

In [5]:
from sklearn.model_selection import train_test_split

TRAIN_SAMPLE = 150000   # None = usar o treino todo

if TRAIN_SAMPLE:
    X_fit, _, y_fit, _ = train_test_split(
        X_train, y_train, train_size=TRAIN_SAMPLE, stratify=y_train, random_state=SEED)
else:
    X_fit, y_fit = X_train, y_train

print("Linhas usadas no treino:", len(X_fit))
print(y_fit.value_counts())

Linhas usadas no treino: 150000
MalwareFamily
Mirai         102775
Unknown        23116
Benign         10541
DarkNexus       8967
Other_rare      2737
Generic         1864
Name: count, dtype: int64


## 3. Pré-processamento e modelos

O pré-processamento é o do Módulo 4 (`log1p` nas colunas enviesadas → imputação pela mediana → `StandardScaler`) e o desequilíbrio trata-se com pesos de classe. Cada modelo tem o seu `Pipeline`, ajustado só no treino.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# colunas enviesadas escolhidas só com o treino
sk = X_fit.skew()
skewed = [c for c in X_fit.columns
          if abs(sk[c]) > 1 and X_fit[c].min() >= 0 and X_fit[c].nunique() > 2]

def prep():
    return [
        ("log", ColumnTransformer([("log1p", FunctionTransformer(np.log1p), skewed)],
                                  remainder="passthrough")),
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]

# XGBoost precisa de classes numéricas
le = LabelEncoder().fit(y_train)
y_fit_enc, y_val_enc = le.transform(y_fit), le.transform(y_val)

modelos = {
    "Logistic Regression": Pipeline(prep() + [("model", LogisticRegression(
        max_iter=300, class_weight="balanced", random_state=SEED))]),
    "Decision Tree": Pipeline(prep() + [("model", DecisionTreeClassifier(
        max_depth=15, class_weight="balanced", random_state=SEED))]),
    "Random Forest": Pipeline(prep() + [("model", RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=SEED, n_jobs=-1))]),
    "XGBoost": Pipeline(prep() + [("model", XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1, tree_method="hist",
        random_state=SEED, n_jobs=-1))]),
}

## 4. Treino e comparação na validação

In [7]:
from sklearn.metrics import f1_score, accuracy_score, classification_report

resultados, previsoes = [], {}

for nome, pipe in modelos.items():
    t0 = time.time()
    if nome == "XGBoost":
        # desequilíbrio: pesos por amostra (equivalente ao class_weight="balanced")
        pipe.fit(X_fit, y_fit_enc, model__sample_weight=compute_sample_weight("balanced", y_fit_enc))
        pred = le.inverse_transform(pipe.predict(X_val))
    else:
        pipe.fit(X_fit, y_fit)
        pred = pipe.predict(X_val)
    previsoes[nome] = pred
    resultados.append({
        "modelo": nome,
        "F1 macro": round(f1_score(y_val, pred, average="macro"), 4),
        "accuracy": round(accuracy_score(y_val, pred), 4),
        "tempo (s)": round(time.time() - t0),
    })
    print(f"{nome}: F1 macro = {resultados[-1]['F1 macro']}  ({resultados[-1]['tempo (s)']}s)")

tabela = pd.DataFrame(resultados).sort_values("F1 macro", ascending=False).reset_index(drop=True)
tabela

C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression: F1 macro = 0.1907  (10s)
Decision Tree: F1 macro = 0.315  (2s)
Random Forest: F1 macro = 0.3501  (5s)
XGBoost: F1 macro = 0.3177  (7s)


,modelo,F1 macro,accuracy,tempo (s)
0,Random Forest,0.3501,0.5585,5
1,XGBoost,0.3177,0.3701,7
2,Decision Tree,0.3150,0.3762,2
3,Logistic Regression,0.1907,0.1890,10


## 5. Detalhe por classe do melhor modelo

In [8]:
melhor = tabela.loc[0, "modelo"]
print("Melhor modelo:", melhor)
print(classification_report(y_val, previsoes[melhor], digits=3, zero_division=0))

Melhor modelo: Random Forest
              precision    recall  f1-score   support

      Benign      0.531     0.892     0.665      5176
   DarkNexus      0.480     0.490     0.485      4407
     Generic      0.004     0.009     0.005       862
       Mirai      0.829     0.591     0.690     50490
  Other_rare      0.003     0.008     0.005       387
     Unknown      0.195     0.351     0.251     11344

    accuracy                          0.559     72666
   macro avg      0.340     0.390     0.350     72666
weighted avg      0.673     0.559     0.595     72666



## 6. Conclusões para o relatório

*(Preenche depois de correres o notebook.)*

- **Melhor modelo na validação:** …  (F1 macro = …)
- **Classes difíceis:** … (ver o detalhe por classe)
- **Comparação:** modelos baseados em árvores vs. Logistic Regression …
- **Limitações:** treino numa amostra; `Other_rare` tem poucos exemplos na validação.
- **Próximo passo (Módulo 7):** matriz de confusão, AUC-ROC, *hyperparameter tuning* do melhor modelo e avaliação final **uma única vez** no teste.

In [9]:

import joblib
from pathlib import Path

# Pasta principal do projeto
pasta_projeto = Path(
    r"C:\Users\User\Documents\grupo4-iaac"
)

# Criar a pasta models, se necessário
pasta_models = pasta_projeto / "models"
pasta_models.mkdir(exist_ok=True)

# Verificar se o Random Forest está treinado
modelo_rf = modelos["Random Forest"]

if not hasattr(modelo_rf, "classes_"):
    raise RuntimeError(
        "O Random Forest ainda não está treinado nesta sessão."
    )

# Guardar o modelo treinado
destino = pasta_models / "random_forest.pkl"

joblib.dump(modelo_rf, destino)

print("Modelo guardado com sucesso!")
print("Localização:", destino)


Modelo guardado com sucesso!
Localização: C:\Users\User\Documents\grupo4-iaac\models\random_forest.pkl


In [10]:
print("Modelos disponíveis:", list(modelos.keys()))
print("Random Forest:", modelos["Random Forest"])

Modelos disponíveis: ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost']
Random Forest: Pipeline(steps=[('log',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('log1p',
                                                  FunctionTransformer(func=<ufunc 'log1p'>),
                                                  ['Protocol Type',
                                                   'Time_To_Live', 'Rate',
                                                   'fin_flag_number',
                                                   'syn_flag_number',
                                                   'rst_flag_number',
                                                   'psh_flag_number',
                                                   'ack_flag_number',
                                                   'ece_flag_number',
                                                   'ack_count', 'syn_count',
                   